# Earth Trend Detective — ETG Trend Microscope (`etg-trend-microscope-v1`) Blackwell B200 Trainer

Trains the **3-in-1 AI Trend Perception Organ** (`etg-trend-microscope-v1`) designed to amplify AI & scientific trend reasoning on multi-scale Earth Trend Graphs rather than replacing forecasters:
1. **Head 1 (`inflection_head`) — Pre-Mann-Kendall Early Onset Radar**: catches emerging slope breaks 3–6 steps before classical rolling Mann-Kendall crosses `|Z| >= 1.96`, ignoring AR(1) red noise.
2. **Head 2 (`coupling_head`) — Directional Lead-Lag (`tau = 0..5`) & Dipole Link Scorer**: distinguishes physical `drives_opposite` / `co_trending` couplings from opposite-hemisphere seasonal illusions and shared secular-drift confounders.
3. **Head 3 (`dealias_head` + `trust_gate`) — Cycle De-Aliasing & Sensor Artifact Filter**: decomposes every signal onto a `[secularShare, oscillationShare, artifactShare]` simplex + `falseTrendProb` while downweighting uncorroborated noisy community sensor glitches.

Workflow: **Transformer Teacher (`d_model=128`, CUDA `bf16` + `torch.compile`) -> Knowledge Distillation -> Portable Weights (`etg-trend-microscope-v1.json`) + ONNX/PyTorch L4 Export (`etg_trend_microscope_v1.{pt,onnx}`)**.

In [1]:
# Cell 1: Unpack Self-Contained ETG Modules & Probe Blackwell B200 / CUDA Runtime
import base64
import importlib
import json
import sys
import zlib
from pathlib import Path

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ML_DIR.mkdir(parents=True, exist_ok=True)
(ML_DIR / "artifacts").mkdir(parents=True, exist_ok=True)

PAYLOAD_B64 = """
...
"""

files = json.loads(zlib.decompress(base64.b64decode("".join(PAYLOAD_B64.split()))).decode("utf-8"))
for rel_name, source in files.items():
    (ML_DIR / rel_name).write_text(source, encoding="utf-8")

if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

for rel_name in files:
    mod_name = Path(rel_name).stem
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print(f"Unpacked {list(files.keys())} -> {ML_DIR}")
try:
    import torch
    print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        cap = torch.cuda.get_device_capability(0)
        bf16_ok = torch.cuda.is_bf16_supported()
        torch.set_float32_matmul_precision("high")
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU: {props.name} | SM {cap[0]}.{cap[1]} | VRAM {vram_gb:.1f} GB | bf16={bf16_ok}")
    else:
        print("WARNING: No CUDA GPU detected; using PyTorch CPU / Portable fallback.")
except Exception as exc:
    print(f"PyTorch unavailable ({exc}); using pure-Python PortableTrendMicroscope.")

Unpacked ['etg_trend_microscope.py', 'harvest_global_multiscale.py'] -> .data/colab_ml
PyTorch: 2.15.0 | CUDA available: False


In [2]:
# Cell 2: Harvest 2000-2025 Global Multi-Scale Bundle & Build Microscope Curriculum Corpus
import os
import sys
import time
from pathlib import Path

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ARTIFACTS_DIR = ML_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from harvest_global_multiscale import (
    build_synthetic_global_multiscale_bundle,
    harvest_live_global_multiscale,
)
from etg_trend_microscope import build_microscope_training_corpus

t0 = time.perf_counter()
live_path = ARTIFACTS_DIR / "etg-ml-v2-live.json"
try:
    if os.environ.get("ETG_OFFLINE") == "1":
        raise RuntimeError("Offline deterministic mode (ETG_OFFLINE=1)")
    live_raw = harvest_live_global_multiscale(
        output_path=str(live_path),
        start_year=2000,
        end_year=2025,
        horizon_steps=1,
    )
    if not live_raw.get("node_ids"):
        raise RuntimeError("Empty node_ids from live harvester")
except Exception as exc:
    print(f"Live harvest fallback triggered ({exc}); building synthetic benchmark bundle.")
    live_raw = build_synthetic_global_multiscale_bundle(
        num_months=84,
        horizon_steps=1,
        seed=42,
    )

corpus = build_microscope_training_corpus(
    live_bundle=live_raw,
    num_synthetic_graphs=320,
    seed=42,
)

meta = live_raw.get("harvester_metadata", {})
print(f"Bundle Mode        : {meta.get('mode', 'synthetic')}")
print(f"Live Sources       : {meta.get('live_sources', [])}")
print(f"Bundle Nodes/Steps : {len(live_raw.get('node_ids', []))} nodes / {len(live_raw.get('time_index', []))} steps")
print(f"Microscope Corpus  : train={len(corpus['train'])} graphs | val={len(corpus['val'])} graphs")
print(f"Corpus preparation finished in {time.perf_counter() - t0:.2f}s")

Live harvest fallback triggered (Offline deterministic mode (ETG_OFFLINE=1)); building synthetic benchmark bundle.
Bundle Mode        : synthetic
Live Sources       : []
Bundle Nodes/Steps : 8 nodes / 84 steps
Microscope Corpus  : train=254 graphs | val=84 graphs
Corpus preparation finished in 2.52s


In [3]:
# Cell 3: Train Blackwell B200 Transformer Teacher (bf16 + torch.compile) & Distill Portable Microscope
import time
import etg_trend_microscope as tm
from etg_trend_microscope import PortableTrendMicroscope, evaluate_microscope_benchmarks

t0 = time.perf_counter()
train_res = None
if getattr(tm, "TORCH_AVAILABLE", False):
    train_res = tm.train_blackwell_microscope_and_distill(
        corpus,
        teacher_epochs=50,
        distill_epochs=65,
        d_model=128,
        lr=1.8e-3,
        compile_model=True,
    )
    teacher_model = train_res["teacher"]
    student_model = train_res["student"]
    portable_model = train_res["portable_model"]
    portable_weights = train_res["portable_weights"]
    benchmarks = train_res["benchmarks"]
    print(f"Backend       : PyTorch Teacher + Distilled Portable ({train_res['device']}, bf16={train_res['bf16']})")
    print(f"Teacher Loss  : start={train_res['teacher_loss_history'][0]:.4f} -> end={train_res['teacher_loss_history'][-1]:.4f}")
    print(f"Distill Loss  : start={train_res['distill_loss_history'][0]:.4f} -> end={train_res['distill_loss_history'][-1]:.4f}")
else:
    teacher_model = None
    student_model = None
    portable_model = PortableTrendMicroscope(hidden_dim=32, seed=42)
    fit_info = portable_model.fit_portable(corpus["train"], epochs=42, lr=0.045, seed=42, batch_size=6)
    benchmarks = evaluate_microscope_benchmarks(portable_model, corpus["val"])
    portable_model.training_summary = {"backend": "portable_adam_cpu_fallback", **fit_info, "benchmarks": benchmarks}
    portable_weights = portable_model.to_weights_dict()
    print("Backend       : Pure-Python PortableTrendMicroscope (CPU fallback)")
    print(f"Loss Curve    : start={fit_info['loss_history'][0]:.4f} -> end={fit_info['loss_history'][-1]:.4f}")

print(f"Training + Distillation completed in {time.perf_counter() - t0:.2f}s")

Backend       : PyTorch Teacher + Distilled Portable (cpu, bf16=False)
Teacher Loss  : start=4.4728 -> end=1.0121
Distill Loss  : start=4.0283 -> end=0.9274
Training + Distillation completed in 75.24s


In [4]:
# Cell 4: 3-Head Holdout Benchmarks vs. Classical Baselines, AI Perception Brief & Artifact Export
import json
from pathlib import Path
import etg_trend_microscope as tm

print("=== ETG Trend Microscope (`etg-trend-microscope-v1`) Holdout Benchmarks ===")
print(
    f"1. Early Onset AUROC     : {benchmarks['early_onset_auroc']:.4f} "f"(vs Classical Mann-Kendall AUROC: {benchmarks['classical_mk_auroc']:.4f})"
)
print(
    f"2. Pre-MK Fresh Break Rec: {benchmarks['pre_mk_recall_on_fresh_breaks']:.4f} "f"(vs Classical Mann-Kendall Recall: {benchmarks['classical_mk_recall_on_fresh_breaks']:.4f})"
)
print(
    f"3. True Link AUROC       : {benchmarks['true_link_auroc']:.4f} "f"(vs Raw Pearson |r| AUROC      : {benchmarks['raw_pearson_auroc']:.4f})"
)
print(f"4. Illusion Rejection    : {benchmarks['illusion_rejection_rate']:.4f}")
print(f"5. De-Alias Simplex MAE  : {benchmarks['dealias_simplex_mae']:.4f}")
print(f"6. False-Trend Detect F1 : {benchmarks['false_trend_f1']:.4f}")

live_scan = portable_model.scan_v2_bundle(live_raw, top_k=8)
print("\n=== Live Bundle AI Perception Brief ===")
print(live_scan["aiPerceptionBrief"])

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ARTIFACTS_DIR = ML_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
json_path = ARTIFACTS_DIR / "etg-trend-microscope-v1.json"
json_path.write_text(json.dumps(portable_weights, indent=2), encoding="utf-8")

pt_path = ARTIFACTS_DIR / "etg_trend_microscope_v1.pt"
onnx_path = ARTIFACTS_DIR / "etg_trend_microscope_v1.onnx"
if teacher_model is not None and getattr(tm, "TORCH_AVAILABLE", False):
    import torch
    torch.save(
        {
            "schema_version": "etg-trend-microscope-v1",
            "teacher_state_dict": teacher_model.state_dict(),
            "student_state_dict": student_model.state_dict() if student_model is not None else None,
            "benchmarks": benchmarks,
        },
        pt_path,
    )
    try:
        tm.export_microscope_onnx(teacher_model, corpus["val"][0], onnx_path)
        print(f"Exported ONNX Teacher  -> {onnx_path}")
    except Exception as exc:
        print(f"ONNX export skipped ({exc})")
    print(f"Exported PyTorch Ckpt  -> {pt_path}")
print(f"Exported JSON Weights  -> {json_path}")
print(
    f"Verified portable schema={portable_weights['schema_version']} | "
    f"size={json_path.stat().st_size:,} bytes | hidden_dim={portable_weights['hidden_dim']}"
)

=== ETG Trend Microscope (`etg-trend-microscope-v1`) Holdout Benchmarks ===
1. Early Onset AUROC     : 0.9706 (vs Classical Mann-Kendall AUROC: 0.4435)
2. Pre-MK Fresh Break Rec: 0.7528 (vs Classical Mann-Kendall Recall: 0.0000)
3. True Link AUROC       : 0.9827 (vs Raw Pearson |r| AUROC      : 0.6808)
4. Illusion Rejection    : 1.0000
5. De-Alias Simplex MAE  : 0.0552
6. False-Trend Detect F1 : 0.9442

=== Live Bundle AI Perception Brief ===
ETG Trend Microscope (etg-trend-microscope-v1) — 8 nodes scanned:
Emerging Onsets: [KNMI Nino 3.4 SST Anomaly] badge=EARLY_ONSET | onset=94% (PRE-MK) | attrib: 67% secular / 16% cycle / 17% noise (false_prob=0%), [Central Amazon Canopy Vapor Pressure Deficit] badge=EARLY_ONSET | onset=81% (PRE-MK) | attrib: 65% secular / 16% cycle / 19% noise (false_prob=0%)
Verified Lagged/Dipole Couplings: Community Snowpack & Soil Flux Node -> Central Amazon Canopy Vapor Pressure Deficit: drives_opposite (P_true=94%, lag=0, P_illusion=17%, raw_r=-0.69, clean_r=